This file is used to train a Seq2seq model for predict latent vetors from eeg signal

## CONFIGURATION

In [1]:
CONFIG = {
    "EEG_data_path": "../data/SEED-DV/Segmented_Rawf_200Hz_2s/sub1.npy",
    "latent_data_path": "1200_latent.npy",
    "test_latent_path": "40classes_latents.pt",
    "save_path": "../checkpoints/seq2seq.pt",

    "video_latent_shape": (250, 12, 4, 36, 64),
    "eeg_t_window": 100,
    "eeg_t_step": 50,

    "d_model": 512,
    "eeg_channels": 62,
    
    "n_head": 4,
    "encoder_layers": 2,
    "decoder_layers": 4,
    "seed": 42,
    "dropout": 0.1,
    "batch_size": 128,
    "learning_rate": 0.001,
    "epochs": 100,

    "max_latent_seq_len": 6,
}

## Set seed

In [ ]:
import torch
import numpy as np
import random
import os

def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

## Define dataset

In [ ]:
from torch.utils.data import DataLoader, Dataset
from einops import rearrange
from sklearn.preprocessing import StandardScaler

class EEGVideoDataset(Dataset):
    """Dataset for pairing EEG sequences with latent video sequences."""
    def __init__(self, eeg_data: torch.Tensor, video_data: torch.Tensor):
        self.eeg = eeg_data
        self.video = video_data
        assert len(self.eeg) == len(self.video), "EEG and video data must have the same number of samples."

    def __len__(self) -> int:
        return len(self.eeg)

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.eeg[item], self.video[item]

def load_and_preprocess_data():
    """
    Loads, preprocesses, and prepares EEG and latent data for training and testing.

    Args:
        cfg (Config): Configuration object with paths and parameters.

    Returns:
        A tuple containing:
        - DataLoader: Dataloader for the training set.
        - torch.Tensor: The preprocessed test EEG data.
        - torch.Tensor: The preprocessed test latent data.
    """
    # Ground truth labels for reordering trials
    GT_LABEL = np.array([[23, 22, 9, 6, 18, 14, 5, 36, 25, 19, 28, 35, 3, 16, 24, 40, 15, 27, 38, 33, 34, 4, 39, 17, 1, 26, 20, 29, 13, 32, 37, 2, 11, 12, 30, 31, 8, 21, 7, 10], [27, 33, 22, 28, 31, 12, 38, 4, 18, 17, 35, 39, 40, 5, 24, 32, 15, 13, 2, 16, 34, 25, 19, 30, 23, 3, 8, 29, 7, 20, 11, 14, 37, 6, 21, 1, 10, 36, 26, 9], [15, 36, 31, 1, 34, 3, 37, 12, 4, 5, 21, 24, 14, 16, 39, 20, 28, 29, 18, 32, 2, 27, 8, 19, 13, 10, 30, 40, 17, 26, 11, 9, 33, 25, 35, 7, 38, 22, 23, 6], [16, 28, 23, 1, 39, 10, 35, 14, 19, 27, 37, 31, 5, 18, 11, 25, 29, 13, 20, 24, 7, 34, 26, 4, 40, 12, 8, 22, 21, 30, 17, 2, 38, 9, 3, 36, 33, 6, 32, 15], [18, 29, 7, 35, 22, 19, 12, 36, 8, 15, 28, 1, 34, 23, 20, 13, 37, 9, 16, 30, 2, 33, 27, 21, 14, 38, 10, 17, 31, 3, 24, 39, 11, 32, 4, 25, 40, 5, 26, 6], [29, 16, 1, 22, 34, 39, 24, 10, 8, 35, 27, 31, 23, 17, 2, 15, 25, 40, 3, 36, 26, 6, 14, 37, 9, 12, 19, 30, 5, 28, 32, 4, 13, 18, 21, 20, 7, 11, 33, 38], [38, 34, 40, 10, 28, 7, 1, 37, 22, 9, 16, 5, 12, 36, 20, 30, 6, 15, 35, 2, 31, 26, 18, 24, 8, 3, 23, 19, 14, 13, 21, 4, 25, 11, 32, 17, 39, 29, 33, 27]])
    
    # --- Load Data ---
    eeg_data = np.load(CONFIG["EEG_data_path"]) # (5, 50, 62, 400)
    latent_data = np.load(CONFIG["latent_data_path"]) # (250, 12, 4, 36, 64)
    train_latent_data = latent_data[:220, ...] # (220, 12, 4, 36, 64)
    test_latent_data = latent_data[220:, ...] # (30, 12, 4, 36, 64)
    

    eeg_data = torch.from_numpy(eeg_data)
    windowed_eeg = eeg_data.unfold(dimension=-1, size=CONFIG["eeg_t_window"], step=CONFIG["eeg_t_step"]) # (5, 50, 62, 400) -> (5, 50, 62, 7, 100)
    windowed_eeg = rearrange(windowed_eeg, 'g v c w t -> (g v) c w t') # (250, 62, 7, 100)
    v, c, w, t = windowed_eeg.shape # (250, 62, 7, 100)
    
    # --- Split Train/Test and Reshape ---
    train_eeg = windowed_eeg[:220, ...] # (220, 62, 7, 100)
    test_eeg = windowed_eeg[220:, ...] # (30, 62, 7, 100)
    train_eeg = rearrange(train_eeg, 'v c w t -> (v c) (w t)') # (220*62, 7*100) -> (13640, 700)
    test_eeg = rearrange(test_eeg, 'v c w t -> (v c) (w t)') # (30*62, 7*100) -> (1860, 700)

    # Normalize EEG Data
    scaler = StandardScaler()
    train_eeg = scaler.fit_transform(train_eeg)
    test_eeg = scaler.transform(test_eeg)

    train_eeg = rearrange(train_eeg, '(v c) (w t) -> v w c t', w = w, c = c, t = t)
    test_eeg = rearrange(test_eeg, '(v c) (w t) -> v w c t', w = w, c = c, t = t)

    print(f"Train EEG shape: {train_eeg.shape}")
    print(f"Train Latent shape: {train_latent_data.shape}")
    print(f"Test EEG shape: {test_eeg.shape}")
    print(f"Test Latent shape: {test_latent_data.shape}")

    # --- Create DataLoader ---
    train_dataset = EEGVideoDataset(train_eeg, train_latent_data)
    train_dataloader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)

    return train_dataloader, test_eeg, test_latent_data

## EEG embedding

In [ ]:
import torch.nn as nn

class MyEEGNet_embedding(nn.Module):
    """
    EEGNet-based feature extractor to generate embeddings from raw EEG signals.

    This network follows the EEGNet architecture with temporal, depthwise, and
    separable convolutions to learn features from EEG data.

    Args:
        d_model (int): The dimensionality of the output embedding.
        C (int): The number of EEG channels.
        T (int): The number of time points in the EEG window.
        F1, D, F2 (int): Hyperparameters for the number of filters and depth
                         multiplier in the convolutional layers.
        cross_subject (bool): If True, uses a lower dropout rate suitable for
                              cross-subject generalization.
    """
    def __init__(self, d_model: int = 128, C: int = 62, T: int = 200, F1: int = 16, 
                 D: int = 4, F2: int = 16, cross_subject: bool = False):
        super().__init__()
        self.drop_out = 0.25 if cross_subject else 0.5
        
        # Temporal Convolution
        self.block_1 = nn.Sequential(
            nn.ZeroPad2d((31, 32, 0, 0)),
            nn.Conv2d(1, F1, (1, 64), bias=False),
            nn.BatchNorm2d(F1)
        )
        
        # Depthwise Convolution
        self.block_2 = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (C, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(self.drop_out)
        )
        
        # Separable Convolution
        self.block_3 = nn.Sequential(
            nn.ZeroPad2d((7, 8, 0, 0)),
            nn.Conv2d(F1 * D, F1 * D, (1, 16), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(self.drop_out)
        )
        
        # Final projection to d_model
        # Note: The input size 48 depends on T. For T=100, Pool(4) -> 25, Pool(8) -> 3.
        # F2 * (T // 4 // 8) = 16 * (100 // 32) = 16 * 3 = 48
        final_conv_output_size = F2 * (T // 32) 
        self.embedding = nn.Linear(final_conv_output_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the EEGNet embedding.

        Args:
            x (torch.Tensor): Input EEG data of shape (batch, 1, C, T).

        Returns:
            torch.Tensor: Embedded EEG features of shape (batch, d_model).
        """
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.block_3(x)
        x = x.view(x.shape[0], -1)
        x = self.embedding(x)
        return x

## Position encoding

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Injects positional information into the input embeddings.

    This implementation is based on the "Attention Is All You Need" paper.

    Args:
        d_model (int): The dimensionality of the embeddings.
        dropout (float): The dropout rate.
        max_len (int): The maximum possible sequence length.
    """
    def __init__(self, d_model: int, dropout: float, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Input tensor of shape (batch, seq_len, d_model).

        Returns:
            torch.Tensor: Tensor with added positional information.
        """
        x = x + self.pe[:, :x.size(1)].requires_grad_(False)
        return self.dropout(x)

## Transformer to predict latent vectors

In [ ]:
class LatentPredictionTransformer(nn.Module):
    """
    A Transformer model for predicting a sequence of latent video representations
    from a sequence of EEG window embeddings.

    Args:
        d_model (int): The main dimensionality of the model.
        n_head (int): The number of attention heads.
        encoder_layers (int): The number of layers in the Transformer encoder.
        decoder_layers (int): The number of layers in the Transformer decoder.
        dropout (float): The dropout rate.
        eeg_channels (int): Number of EEG channels.
        eeg_t_window (int): Time points per EEG window.
    """
    def __init__(self, d_model: int, n_head: int, encoder_layers: int, 
                 decoder_layers: int, dropout: float, eeg_channels: int, 
                 eeg_t_window: int):
        super().__init__()
        self.d_model = d_model
        
        # --- Embeddings ---
        # For EEG data (source sequence)
        self.eeg_embedding = MyEEGNet_embedding(d_model=d_model, C=eeg_channels, T=eeg_t_window)
        # For latent video frames (target sequence)
        self.latent_embedding = nn.Linear(4 * 36 * 64, d_model)
        
        # --- Positional Encoding ---
        self.positional_encoding = PositionalEncoding(d_model, dropout=dropout)
        
        # --- Transformer Core ---
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_head, batch_first=True, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=encoder_layers)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=n_head, batch_first=True, dropout=dropout)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_layers)
        
        # --- Output Predictors ---
        # Predicts a single class from the mean of encoder outputs
        self.txt_predictor = nn.Linear(d_model, 13) 
        # Predicts the next latent frame representation
        self.latent_predictor = nn.Linear(d_model, 4 * 36 * 64)

    def forward(self, src_eeg: torch.Tensor, tgt_latent: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass for training (uses teacher forcing).

        Args:
            src_eeg (torch.Tensor): Source EEG data. 
                Shape: (batch, seq_len_eeg, channels, time_points).
            tgt_latent (torch.Tensor): Target latent sequence (shifted right).
                Shape: (batch, seq_len_latent, 4, 36, 64).

        Returns:
            A tuple containing:
            - torch.Tensor: Text/class prediction. Shape: (batch, num_classes).
            - torch.Tensor: Predicted latent sequence. Shape: (batch, seq_len_latent, 4, 36, 64).
        """
        # 1. Process source (EEG) sequence
        b, seq_len_eeg, c, t = src_eeg.shape
        src_embedded = self.eeg_embedding(src_eeg.view(b * seq_len_eeg, 1, c, t))
        src_embedded = src_embedded.view(b, seq_len_eeg, self.d_model)
        src = self.positional_encoding(src_embedded)
        
        # 2. Process target (latent) sequence
        b, seq_len_latent, c, h, w = tgt_latent.shape
        tgt_flattened = tgt_latent.view(b, seq_len_latent, -1)
        tgt_embedded = self.latent_embedding(tgt_flattened)
        tgt = self.positional_encoding(tgt_embedded)

        # 3. Generate mask for the decoder
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len_latent).to(src.device)
        
        # 4. Run through Transformer
        encoder_output = self.transformer_encoder(src)
        decoder_output = self.transformer_decoder(tgt, encoder_output, tgt_mask=tgt_mask)
        
        # 5. Generate predictions
        txt_prediction = self.txt_predictor(torch.mean(encoder_output, dim=1))
        latent_prediction_flat = self.latent_predictor(decoder_output)
        latent_prediction = latent_prediction_flat.view(b, seq_len_latent, 4, 36, 64)
        
        return txt_prediction, latent_prediction

    def generate(self, src_eeg: torch.Tensor, max_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Auto-regressively generate a latent sequence for inference.

        Args:
            src_eeg (torch.Tensor): Source EEG data.
                Shape: (batch, seq_len_eeg, channels, time_points).
            max_len (int): The maximum number of latent frames to generate.

        Returns:
            A tuple containing:
            - torch.Tensor: Text/class prediction. Shape: (batch, num_classes).
            - torch.Tensor: Generated latent sequence. Shape: (batch, max_len, 4, 36, 64).
        """
        self.eval()
        device = src_eeg.device
        
        # 1. Process source (EEG) sequence
        b, seq_len_eeg, c, t = src_eeg.shape
        src_embedded = self.eeg_embedding(src_eeg.view(b * seq_len_eeg, 1, c, t))
        src_embedded = src_embedded.view(b, seq_len_eeg, self.d_model)
        src = self.positional_encoding(src_embedded)
        
        # 2. Run encoder
        encoder_output = self.transformer_encoder(src)
        
        # 3. Text prediction from encoder output
        txt_prediction = self.txt_predictor(torch.mean(encoder_output, dim=1))

        # 4. Auto-regressive decoding
        # Start with a zero tensor as the "start-of-sequence" token
        decoder_input = torch.zeros((b, 1, self.d_model), device=device)
        
        generated_sequence = []

        for _ in range(max_len):
            # Generate mask for the current sequence length
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(decoder_input.size(1)).to(device)
            
            # Get model output for the current sequence
            decoder_output = self.transformer_decoder(decoder_input, encoder_output, tgt_mask=tgt_mask)
            
            # Predict the next latent frame from the last time step
            next_latent_flat = self.latent_predictor(decoder_output[:, -1:])
            
            # Append the predicted frame (in its raw, un-embedded form) to our results
            generated_sequence.append(next_latent_flat.view(b, 1, 4, 36, 64))

            # Embed the prediction and append it to the decoder input for the next step
            next_latent_embedded = self.latent_embedding(next_latent_flat)
            decoder_input = torch.cat([decoder_input, next_latent_embedded], dim=1)

        output_latents = torch.cat(generated_sequence, dim=1)
        return txt_prediction, output_latents

## Inference function

In [ ]:
def run_inference(model: nn.Module, test_eeg: torch.Tensor, device: torch.device, 
                  max_len: int) -> np.ndarray:
    """
    Runs inference on the test set and returns the generated latent sequence.
    
    Returns:
        np.ndarray: The predicted latent sequences.
    """
    model.eval()
    with torch.no_grad():
        test_eeg = test_eeg.float().to(device)
        _, latent_out = model.generate(test_eeg, max_len=max_len)
    
    return latent_out.cpu().numpy()

## Train function

In [ ]:
def train_one_epoch(model: nn.Module, dataloader: DataLoader, loss_fn: nn.Module,
                    optimizer: torch.optim.Optimizer, scheduler: Any, device: torch.device) -> float:
    """
    Performs one full training pass over the dataset.

    Returns:
        float: The average loss for the epoch.
    """
    model.train()
    total_loss = 0.0
    for eeg_seq, video_seq in dataloader:
        eeg_seq = eeg_seq.float().to(device)
        video_seq = video_seq.float().to(device)

        # Prepare target sequences for teacher forcing
        # Input to the decoder is the sequence shifted right, with a start token
        start_token = torch.zeros_like(video_seq[:, :1, ...]) # (b, 1, c, h, w)
        target_video_for_input = torch.cat([start_token, video_seq[:, :-1, ...]], dim=1)
        
        # Ground truth for loss calculation is the original sequence
        ground_truth_video = video_seq

        optimizer.zero_grad()
        
        _, pred_video_seq = model(eeg_seq, target_video_for_input)
        
        loss = loss_fn(pred_video_seq, ground_truth_video)
        loss.backward()
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

## Define main function

In [ ]:
def main():
    """Main function to run the training and inference pipeline."""
    cfg = Config()
    cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # --- Data Preparation ---
    train_loader, test_eeg, _ = load_and_preprocess_data(cfg)

    # --- Model Initialization ---
    model = LatentPredictionTransformer(
        d_model=cfg.D_MODEL,
        n_head=cfg.N_HEAD,
        encoder_layers=cfg.ENCODER_LAYERS,
        decoder_layers=cfg.DECODER_LAYERS,
        dropout=cfg.DROPOUT,
        eeg_channels=cfg.EEG_CHANNELS,
        eeg_t_window=cfg.EEG_T_WINDOW
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS * len(train_loader))
    loss_fn = nn.MSELoss()

    # --- Training Loop ---
    print("Starting training...")
    for epoch in range(cfg.EPOCHS):
        epoch_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, scheduler, device)
        print(f"Epoch {epoch+1}/{cfg.EPOCHS}, Loss: {epoch_loss:.6f}")

    # --- Inference ---
    print("Training finished. Running inference...")
    predicted_latents = run_inference(model, test_eeg, device, max_len=cfg.MAX_LATENT_SEQ_LEN)
    
    print(f"Shape of predicted latents: {predicted_latents.shape}")
    output_path = 'latent_out_block7_40_classes.npy'
    torch.save(output_path, predicted_latents)
    print(f"Saved predicted latents to {output_path}")

    # --- Save Model ---
    model_save_path = cfg.OUTPUT_DIR / 'seq2seqmodel.pt'
    torch.save({'state_dict': model.state_dict()}, model_save_path)
    print(f"Saved model state_dict to {model_save_path}")

## Start!

In [ ]:
if __name__ == "__main__":
    main()

---
---
---

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from einops import rearrange
from tqdm import tqdm
from pathlib import Path
from typing import Tuple, Dict, Any

# --- Configuration ---

class Config:
    """Configuration class for hyperparameters and file paths."""
    # Data Paths
    DATA_DIR = Path('../data/SEED-DV/Segmented_Rawf_200Hz_2s/')
    EEG_DATA_PATH = DATA_DIR / 'sub1.npy'
    LATENT_DATA_PATH = Path('1200_latent.npy')
    TEST_LATENT_PATH = Path('40classes_latents.pt')
    OUTPUT_DIR = Path('../checkpoints/')
    
    # Model Hyperparameters
    D_MODEL = 512
    EEG_CHANNELS = 62
    EEG_T_WINDOW = 100
    N_HEAD = 4
    ENCODER_LAYERS = 2
    DECODER_LAYERS = 4
    DROPOUT = 0.1
    
    # Training Hyperparameters
    EPOCHS = 200
    BATCH_SIZE = 32
    LEARNING_RATE = 5e-4
    
    # Sequence Length
    MAX_LATENT_SEQ_LEN = 6 # The number of latent frames to predict
